In [0]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"


In [0]:
print(bronze_schema, silver_schema, gold_schema)

dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:

# ============================================================
# LOAD SILVER ORDERS
# ============================================================

df_silver = spark.sql(
    f"""
    SELECT *
    FROM {catalog}.{silver_schema}.{data_source}
    """
)

display(df_silver.limit(10))

In [0]:
# ============================================================
# LOAD DIMENSIONS / FACT REFERENCES
# ============================================================

df_products = spark.table(
    f"{catalog}.{gold_schema}.dim_products"
).select(
    "product_id",
    "price"
)
df_products.show()

df_payments = spark.table(
    f"{catalog}.{gold_schema}.fact_payments"
).select(
    "payment_id",
    "payment_status"
)
df_payments.show()

df_shipments = spark.table(
    f"{catalog}.{gold_schema}.fact_shipments"
).select(
    "shipment_id",
    "shipment_status"
)
df_shipments.show()


In [0]:
# ============================================================
# ENRICH ORDERS
# ============================================================

df_gold = (

    df_silver

    # PRODUCT JOIN
    .join(
        df_products,
        on="product_id",
        how="left"
    )

    # PAYMENT JOIN
    .join(
        df_payments,
        on="payment_id",
        how="left"
    )

    # SHIPMENT JOIN
    .join(
        df_shipments,
        on="shipment_id",
        how="left"
    )
)

display(df_gold.limit(20))

In [0]:
df_gold = (

    df_gold

    # ORDER AMOUNT
    .withColumn(
        "order_amount",
        F.col("quantity") * F.col("price")
    )

    # ORDER DATE
    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )

    # AUDIT COLUMN
    .withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )
)
display(df_gold.limit(5))

In [0]:
# ============================================================
# FINAL GOLD FACT PROJECTION
# ============================================================

df_gold = df_gold.select(

    "order_id",
    "customer_id",
    "product_id",
    "payment_id",
    "shipment_id",

    "quantity",
    "price",
    "order_amount",

    "payment_status",
    "shipment_status",

    "order_timestamp",
    "order_date",

    "processed_timestamp"
)


display(df_gold.limit(10))


In [0]:
# ============================================================
# DEFENSIVE DEDUPLICATION
# ============================================================

df_gold = df_gold.dropDuplicates([
    "order_id",
    "product_id"
])



In [0]:
# ============================================================
# GOLD FACT WRITE LOGIC
# ============================================================

if not spark.catalog.tableExists(
    f"{catalog}.{gold_schema}.fact_{data_source}"
):

    (
        df_gold.write
            .format("delta")

            .option(
                "delta.enableChangeDataFeed",
                "true"
            )

            .mode("overwrite")

            .saveAsTable(
                f"{catalog}.{gold_schema}.fact_{data_source}"
            )
    )

    print(
        f"Successfully created fact_{data_source}"
    )


# ============================================================
# INCREMENTAL MERGE
# ============================================================

else:

    print(
        f"Running incremental MERGE for fact_{data_source}"
    )

    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{gold_schema}.fact_{data_source}"
    )

    (
        delta_table.alias("target")

        .merge(

            source=df_gold.alias("source"),

            condition="""
                target.order_id = source.order_id
                AND
                target.product_id = source.product_id
            """
        )

        .whenMatchedUpdate(

            condition="""
                NOT (target.customer_id <=> source.customer_id)
                OR NOT (target.payment_id <=> source.payment_id)
                OR NOT (target.shipment_id <=> source.shipment_id)
                OR NOT (target.quantity <=> source.quantity)
                OR NOT (target.price <=> source.price)
                OR NOT (target.order_amount <=> source.order_amount)
                OR NOT (target.payment_status <=> source.payment_status)
                OR NOT (target.shipment_status <=> source.shipment_status)
            """,

            set={

                "customer_id":
                    "source.customer_id",

                "payment_id":
                    "source.payment_id",

                "shipment_id":
                    "source.shipment_id",

                "quantity":
                    "source.quantity",

                "price":
                    "source.price",

                "order_amount":
                    "source.order_amount",

                "payment_status":
                    "source.payment_status",

                "shipment_status":
                    "source.shipment_status",

                "order_timestamp":
                    "source.order_timestamp",

                "order_date":
                    "source.order_date",

                "processed_timestamp":
                    "source.processed_timestamp"
            }
        )

        .whenNotMatchedInsert(

            values={

                "order_id":
                    "source.order_id",

                "customer_id":
                    "source.customer_id",

                "product_id":
                    "source.product_id",

                "payment_id":
                    "source.payment_id",

                "shipment_id":
                    "source.shipment_id",

                "quantity":
                    "source.quantity",

                "price":
                    "source.price",

                "order_amount":
                    "source.order_amount",

                "payment_status":
                    "source.payment_status",

                "shipment_status":
                    "source.shipment_status",

                "order_timestamp":
                    "source.order_timestamp",

                "order_date":
                    "source.order_date",

                "processed_timestamp":
                    "source.processed_timestamp"
            }
        )

        .execute()
    )

    print(
        f"Successfully merged into fact_{data_source}"
    )